# HW1 - Recommender Systems Simulation
**AUEB MSc Data Science — Video Games domain (PC & PlayStation)**

We build a synthetic dataset of users rating video games, then use **LDA with anchor words** to recover the 5 user segments purely from rating patterns — no segment labels used during learning.

> See `README.md` for a full explanation of the approach and the pipeline.

In [ ]:
!pip install tomotopy -q

---
## Upload `game_titles.csv`

Run the cell below and select `game_titles.csv` from your computer.
This is the pool of ~560 real game titles — upload it once per session.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select game_titles.csv from your computer

---
## Imports & Setup

In [ ]:
import random
import csv
import re
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Gamer',
    2: 'Console Gamer',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

_titles_df = pd.read_csv('game_titles.csv')
GAME_TITLES = _titles_df['title'].dropna().str.strip().tolist()
print(f'Loaded {len(GAME_TITLES)} game titles from game_titles.csv')


def _title_to_token(title: str) -> str:
    """Converts a game title to a safe LDA token (no spaces or special chars)."""
    safe = re.sub(r'[^A-Za-z0-9]+', '_', title)
    return safe.strip('_')

---
## 1. `generate_entities()` — Video Games

Samples 300 real game titles from `game_titles.csv` and assigns synthetic attributes (platform, genre, price, Metacritic score, PEGI rating, etc.) drawn from Gaussian distributions.

In [ ]:
@dataclass
class VideoGame:
    title: str             # display name (e.g. "Elden Ring")
    token: str             # safe LDA token (e.g. "Elden_Ring")
    platform: str          # 'PC', 'PS', or 'BOTH'
    genre: str
    price_eur: float
    metacritic: int        # 0-100
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool     # True when platform != 'BOTH'
    age_rating: str        # PEGI: '3','7','12','16','18'
    release_year: int

In [ ]:
def generate_entities(
    game_num: int = 300,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.42, 0.35, 0.23],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (1994, 2024)
) -> List[VideoGame]:
    """
    Generates game_num synthetic VideoGame objects sampled from GAME_TITLES.

    Platform split default [0.42, 0.35, 0.23]: mostly single-platform exclusives,
    fewer cross-platform (BOTH) titles — reflecting real-world game distribution.
    'BOTH' = playable on PC AND PlayStation; a PC Gamer can access it,
    but a PS-only player cannot play a PC-exclusive (and vice versa).
    """
    titles = random.sample(GAME_TITLES, min(game_num, len(GAME_TITLES)))
    games = []
    for title in titles:
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=title, token=_title_to_token(title),
            platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [ ]:
games = generate_entities(game_num=300)
print(f'Generated {len(games)} games.')
print(f'Sample: {games[0].title!r}  |  platform: {games[0].platform}  |  genre: {games[0].genre}  |  €{games[0].price_eur}  |  Metacritic: {games[0].metacritic}')
print('Platform split:', pd.Series([g.platform for g in games]).value_counts().to_dict())

---
## 2. `generate_users()` — User Segments

Creates 1 000 users (200 per segment). Each user has **personal preferences**: a preferred platform, a list of favorite genres, and a price limit. These drive the rating logic — not just the segment label.

In [ ]:
@dataclass
class User:
    segment: int
    age: int
    gender: str
    preferred_platform: str   # 'PC', 'PS', or 'BOTH'
    favorite_genres: list     # e.g. ['Action', 'RPG']
    price_limit: float        # max price willing to pay (EUR)

In [ ]:
def generate_users_segment1(user_num: int = 350) -> List[User]:
    """Segment 1 - PC Gamer. Plays on PC, likes Action/RPG/Strategy, willing to pay full price."""
    genres = ['Action', 'RPG', 'Strategy', 'Adventure', 'Horror']
    return [User(
        segment=1, age=max(10, int(random.gauss(30, 6))), gender=random.choice(['M', 'F']),
        preferred_platform='PC',
        favorite_genres=random.sample(genres, k=random.randint(2, 3)),
        price_limit=round(random.gauss(60, 15), 2)
    ) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 320) -> List[User]:
    """Segment 2 - Console Gamer. Plays on PS, likes Action/Adventure/Sports, moderate budget."""
    genres = ['Action', 'Adventure', 'Sports', 'Fighting', 'Racing']
    return [User(
        segment=2, age=max(10, int(random.gauss(25, 7))), gender=random.choice(['M', 'F']),
        preferred_platform='PS',
        favorite_genres=random.sample(genres, k=random.randint(2, 3)),
        price_limit=round(random.gauss(50, 12), 2)
    ) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 130) -> List[User]:
    """Segment 3 - Cross-Platform Gamer. Plays on both, broad genre tastes, high budget."""
    genres = ['Action', 'RPG', 'Adventure', 'Strategy', 'Simulation', 'Puzzle']
    return [User(
        segment=3, age=max(10, int(random.gauss(22, 5))), gender=random.choice(['M', 'F']),
        preferred_platform='BOTH',
        favorite_genres=random.sample(genres, k=random.randint(3, 4)),
        price_limit=round(random.gauss(70, 10), 2)
    ) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 100) -> List[User]:
    """Segment 4 - Budget Gamer. PC or PS (mostly), price is the only thing that matters."""
    genres = ['Action', 'RPG', 'Sports', 'Puzzle', 'Simulation', 'Racing', 'Horror']
    return [User(
        segment=4, age=max(10, int(random.gauss(20, 8))), gender=random.choice(['M', 'F']),
        # Budget gamers are mostly PC or console, rarely cross-platform
        preferred_platform=random.choices(['PC', 'PS', 'BOTH'], weights=[0.45, 0.45, 0.10])[0],
        favorite_genres=random.sample(genres, k=random.randint(2, 4)),
        price_limit=round(max(5.0, random.gauss(20, 5)), 2)
    ) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 100) -> List[User]:
    """Segment 5 - Casual/Family Gamer. Leans console (PS), family-friendly, low budget."""
    genres = ['Puzzle', 'Simulation', 'Adventure', 'Sports', 'Racing']
    return [User(
        segment=5, age=max(10, int(random.gauss(38, 10))), gender=random.choice(['M', 'F']),
        # Family gamers lean towards console (PS), some on PC, rarely cross-platform
        preferred_platform=random.choices(['PC', 'PS', 'BOTH'], weights=[0.25, 0.65, 0.10])[0],
        favorite_genres=random.sample(genres, k=random.randint(1, 3)),
        price_limit=round(max(5.0, random.gauss(30, 10)), 2)
    ) for _ in range(user_num)]

In [ ]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a shuffled population of users across 5 segments with realistic proportions.

    Segment counts (default total = 1000):
        Seg 1 - PC Gamer       : 350  (most common)
        Seg 2 - Console Gamer  : 320
        Seg 3 - Cross-Platform : 130  (rarer - not everyone owns both)
        Seg 4 - Budget Gamer   : 100
        Seg 5 - Casual/Family  : 100
    """
    total = user_num
    n1 = round(total * 0.35)
    n2 = round(total * 0.32)
    n3 = round(total * 0.13)
    n4 = round(total * 0.10)
    n5 = total - n1 - n2 - n3 - n4
    users = (
        generate_users_segment1(n1) + generate_users_segment2(n2) +
        generate_users_segment3(n3) + generate_users_segment4(n4) +
        generate_users_segment5(n5)
    )
    random.shuffle(users)
    return users

In [ ]:
users = generate_users(user_num=1000)

counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {cnt} users')

# Show a sample user from each segment to verify preferences
print()
seen = set()
for u in users:
    if u.segment not in seen:
        print(f'  Seg {u.segment} sample → platform: {u.preferred_platform:4s} | '
              f'genres: {u.favorite_genres} | price_limit: €{u.price_limit}')
        seen.add(u.segment)
    if len(seen) == 5:
        break

---
## 3. `generate_ratings()` — Binary Ratings

Samples 10 000 (user, game) pairs. Each segment helper applies its **hard rule** (platform, Metacritic, price, age rating) combined with the user's **personal preferences** (favorite genres, price limit) to decide +1 / −1. Every rating is then independently flipped with `noise=10%`.

In [ ]:
def _make_row(user, game, rating, reason):
    """Helper: flattens a user-game pair into a CSV row dict."""
    return {
        'segment': user.segment, 'age': user.age, 'gender': user.gender,
        'preferred_platform': user.preferred_platform,
        'favorite_genres': '|'.join(user.favorite_genres),
        'price_limit': user.price_limit,
        'game': game.token, 'platform': game.platform, 'genre': game.genre,
        'price_eur': game.price_eur, 'metacritic': game.metacritic,
        'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
        'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
        'release_year': game.release_year, 'rating': rating, 'reason': reason
    }

In [ ]:
def generate_ratings_segment1(users, games, pairs, noise):
    """Segment 1 - PC Gamer. Hard rule: PC/BOTH + Metacritic >= 60. Boost if favorite genre."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PC', 'BOTH'):
            rating, reason = -1, 'Not on PC'
        elif game.metacritic < 60:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'PC/BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'PC/BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment2(users, games, pairs, noise):
    """Segment 2 - Console Gamer. Hard rule: PS/BOTH + Metacritic >= 55. Boost if favorite genre."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PS', 'BOTH'):
            rating, reason = -1, 'Not on PlayStation'
        elif game.metacritic < 55:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'PS/BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'PS/BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment3(users, games, pairs, noise):
    """Segment 3 - Cross-Platform. Hard rule: BOTH + Metacritic >= 45. Boost if favorite genre."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform != 'BOTH':
            rating, reason = -1, 'Not on both platforms'
        elif game.metacritic < 45:
            rating, reason = -1, 'Metacritic too low'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'BOTH + good Metacritic + favorite genre'
        else:
            rating, reason = 1, 'BOTH + good Metacritic'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment4(users, games, pairs, noise):
    """Segment 4 - Budget Gamer. Hard rule: price <= user.price_limit (~€20 avg)."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.price_eur <= user.price_limit:
            rating, reason = 1, f'Within budget (€{game.price_eur} ≤ €{user.price_limit})'
        else:
            rating, reason = -1, f'Too expensive (€{game.price_eur} > €{user.price_limit})'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment5(users, games, pairs, noise):
    """Segment 5 - Casual/Family. Hard rule: age_rating in ('3','7'). Boost if favorite genre."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.age_rating not in ('3', '7'):
            rating, reason = -1, 'Age rating too high'
        elif game.genre in user.favorite_genres:
            rating, reason = 1, 'Family-friendly + favorite genre'
        else:
            rating, reason = 1, 'Family-friendly'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.10,
    output_file: str = 'ratings.csv'
) -> None:
    """Samples n_ratings (user, game) pairs, assigns binary ratings, writes to CSV."""
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {1: generate_ratings_segment1, 2: generate_ratings_segment2,
                3: generate_ratings_segment3, 4: generate_ratings_segment4,
                5: generate_ratings_segment5}

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise))
    random.shuffle(all_rows)

    fieldnames = [
        'segment', 'age', 'gender', 'preferred_platform', 'favorite_genres', 'price_limit',
        'game', 'platform', 'genre', 'price_eur', 'metacritic', 'avg_playtime_h',
        'is_multiplayer', 'is_exclusive', 'age_rating', 'release_year', 'rating', 'reason'
    ]
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Ratings saved to {output_file}')
    print(f'Total: {total:,}  |  Likes: {positive:,} ({positive/total:.1%})  |  Noise: {noise:.0%}')

In [ ]:
generate_ratings(users, games, n_ratings=10000, noise=0.10, output_file='ratings.csv')

df = pd.read_csv('ratings.csv')
print()
print('Like rate per segment:')
for seg, grp in df.groupby('segment'):
    print(f'  Seg {seg} - {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

In [ ]:
df.head(10)

The `reason` column shows exactly why the rating was assigned. Notice how `[NOISE]` appears on ~10% of rows where the rating was flipped.

---
## 4. `learn_segments()` — LDA with Anchor Words

Each user becomes a **document**; each rated game becomes a **token** like `Elden_Ring_PC_RPG_LIKE`. LDA finds 5 latent topics — we guide it with **anchor words** (tokens known to belong to each segment) so topics align with the true segments. The confusion matrix at the end tells us how well it worked.

In [ ]:
def learn_segments(
    ratings_file: str = 'ratings.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> pd.DataFrame:
    """Trains LDA on user-document rating corpus; returns true segment vs LDA topic cross-tab."""
    df = pd.read_csv(ratings_file)
    game_lookup = {g.token: g for g in games} if games else {}

    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [
            f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
            for _, r in grp.iterrows()
        ]
        if tokens:
            user_docs[uid] = tokens

    print(f'Users (documents): {len(user_docs)}  |  Avg tokens/user: {np.mean([len(v) for v in user_docs.values()]):.1f}')

    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            g_token = t.rsplit('_', 3)[0]
            if g_token not in game_lookup:
                continue
            g = game_lookup[g_token]
            if g.price_eur <= 25 and t.endswith('LIKE'):
                anchor_budget.append(t)
            if g.age_rating in ('3', '7') and t.endswith('LIKE'):
                anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'Anchor words  →  PC: {len(anchor_pc)}, PS: {len(anchor_ps)}, '
          f'BOTH: {len(anchor_both)}, Budget: {len(anchor_budget)}, Casual: {len(anchor_casual)}')

    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)

    print(f'Training LDA  ({n_iter} iterations)...')
    lda.train(n_iter)
    print(f'Done  |  log-likelihood per word: {lda.ll_per_word:.4f}')

    print()
    for tid in range(lda.k):
        top_words = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c  = sum(1 for w in top_words if '_PC_' in w)
        ps_c  = sum(1 for w in top_words if '_PS_' in w)
        bot_c = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dominant = max({'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c},
                       key=lambda x: {'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c}[x])
        genres = [p[2] for w in top_words if len(p := w.rsplit('_', 3)) == 4]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'Topic {tid}  |  platform: {dominant:4s}  genre: {top_genre:12s}  '
              f'LIKE: {like_c}/{top_n_words}  top: {top_words[0]}')

    uid_list = list(user_docs.keys())
    rows = [
        {'true_segment': int(uid_list[i].split('_')[0]),
         'lda_topic': int(np.argmax(doc.get_topic_dist()))}
        for i, doc in enumerate(lda.docs)
    ]
    ct = pd.crosstab(
        pd.DataFrame(rows)['true_segment'],
        pd.DataFrame(rows)['lda_topic'],
        rownames=['True Segment'], colnames=['LDA Topic']
    )
    return ct

In [ ]:
ct = learn_segments(ratings_file='ratings.csv', games=games, k=5, n_iter=500)

---
## Results — Confusion Matrix

Rows = true segment, columns = LDA topic the user was assigned to. A strong diagonal means LDA recovered the segments well.

In [ ]:
ct_full = ct.reindex(index=range(1, 6), columns=range(5), fill_value=0)

row_labels = [f'Seg {i} - {SEG_NAMES[i]}' for i in range(1, 6)]
col_labels  = [f'Topic {j}' for j in range(5)]
cm_df = pd.DataFrame(ct_full.values, index=row_labels, columns=col_labels)

print('Confusion Matrix  (rows = true segment, columns = LDA topic)\n')
print(cm_df.to_string())
print()
for i, row in enumerate(ct_full.values):
    total = row.sum(); best = row.max(); seg = i + 1
    print(f'  Seg {seg} - {SEG_NAMES[seg]:25s}: {best}/{total} ({best/total:.0%}) → Topic {row.argmax()}')

**Reading the matrix:** each row is a true segment; each column is the LDA topic most users were assigned to. A strong diagonal means LDA recovered the segments well. Segments 1-3 (platform-based) tend to be easier to separate; Segments 4-5 (price / age rating) can overlap more since those signals are weaker in the token vocabulary.